In [ ]:
import trimesh
import os
import numpy as np
import torch as th
from superfit.optim.curvature import get_points_and_weights
from geolipi.torch_compute import Sketcher, recursive_evaluate
from superfit.utils.config import AlgorithmConfig as AlgConf
from superfit.utils.mesh_preprocess import process_mesh_to_sdf


In [ ]:
# Add mesh path here. 
target_mesh_path = ""
if not os.path.exists(target_mesh_path):
    raise ValueError(f"Mesh path {target_mesh_path} does not exist.")
sketcher = Sketcher(device="cuda", n_dims=3, resolution=256)

In [ ]:
from superfit.utils.mesh_preprocess import cd_based_process_mesh_to_sdf
target_mesh, target_sdf, _ = cd_based_process_mesh_to_sdf(target_mesh_path, sketcher)

In [ ]:
from superfit.algos.decompose_msd import msd

# Typical setting
# decompose_config = {
#     "min_eroded_part_size_ratio": 0.005,
#     "min_part_size_ratio": 0.0005,
#     "size_limit": 20,
#     "max_msd_iter": 5,
# }
# MSD to INF
decompose_config = {
        "min_eroded_part_size_ratio": 0.001,
        "min_part_size_ratio": 0.0001,
        "size_limit": 40,
        "max_msd_iter": 20,
    }

pruned_parts, _ = msd(target_sdf, sketcher, **decompose_config)



In [ ]:
# Visualize:
from superfit.utils.mesh_preprocess import sdf_to_mesh
scene = trimesh.Scene()

for part in pruned_parts:
    mesh = sdf_to_mesh(part, sketcher)

    # trimesh expects uint8 RGBA colors, one color per face (or per vertex)
    rgb = (np.random.rand(3) * 255).astype(np.uint8)
    rgba = np.concatenate([rgb, np.array([255], dtype=np.uint8)])
    face_colors = np.tile(rgba, (len(mesh.faces), 1))
    mesh.visual.face_colors = face_colors

    scene.add_geometry(mesh)

# Save:
scene.show()